In [0]:
# ============================================================
# CHESS HISTORIC GAMES - BRONZE LAYER CONFIGURATION
# ============================================================

# Create the grandmasters catalog and bronze schema if they do not exist
spark.sql("CREATE CATALOG IF NOT EXISTS grandmasters")
spark.sql("CREATE DATABASE IF NOT EXISTS grandmasters.bronze")

# Lichess API - free, no credentials required
# Returns game data in NDJSON format with all fields needed:
# WhiteElo, BlackElo, ECO, Opening, TimeControl, UTCDate, Result, RatingDiff
LICHESS_API_URL = "https://lichess.org/api/games/user"

# Player tiers - from elite GMs to club-level players
# This mix enables analysis: "What separates each tier?"
PLAYERS = {
    # Tier 1: Elite GMs (2700+) - Benchmark
    "elite_gm": [
        "DrNykterstein",        # Magnus Carlsen (~3200)
        "FairChess_on_YouTube", # Alireza Firouzja (~2800)
        "nihalsarin2004",       # Nihal Sarin (~2700)
    ],
    # Tier 2: GMs & Strong IMs (2400-2600)
    "gm_im": [
        "penguingm1",           # Andrew Tang (GM, speed specialist)
        "DanielNaroditsky",     # Daniel Naroditsky (GM)
        "Msb2",                 # Abhimanyu Mishra (youngest GM)
        "EricRosen",            # Eric Rosen (IM ~2400)
    ],
    # Tier 3: Rising / Upcoming (1800-2200)
    "rising": [
        "vihaan2107",           # Upcoming player
        "rebsamhsif",           # Active rising player
        "ajedrezconhumor",      # Active ~2000 player
    ],
    # Tier 4: Club level (1400-1800)
    "club": [
        "chess_noob_2024",      # Club player
        "blitz_learner",        # Active casual
        "improving_everyday",   # Improving player
    ],
}

# Flatten for fetching
ALL_PLAYERS = []
for tier, players in PLAYERS.items():
    ALL_PLAYERS.extend([(p, tier) for p in players])

# Max games to fetch per player
MAX_GAMES_PER_PLAYER = 200

# Delta Table
BRONZE_TABLE = "grandmasters.bronze.historic_feed"

# Development flag
# True  -> Clear Bronze before every run
# False -> Keep appending to Bronze
DEVELOPMENT_MODE = True

print(f"API URL          : {LICHESS_API_URL}")
print(f"Grandmasters     : {GRANDMASTERS}")
print(f"Games per player : {MAX_GAMES_PER_PLAYER}")
print(f"Bronze table     : {BRONZE_TABLE}")
print(f"Development mode : {DEVELOPMENT_MODE}")

In [0]:
# ============================================================
# DOWNLOAD HISTORIC GAMES FROM LICHESS API
#
# Purpose:
# Fetch rated games for each grandmaster via the Lichess API.
# Each game is returned as a JSON object (NDJSON stream).
# No credentials required - Lichess is fully open.
# ============================================================

import requests
import json
from datetime import datetime, timezone


def fetch_player_games(username: str, max_games: int) -> list:
    """
    Downloads historic games for a player from the Lichess API.
    Returns a list of raw JSON game objects.
    """
    url = f"{LICHESS_API_URL}/{username}"
    params = {
        "max": max_games,
        "rated": "true",
        "opening": "true",
        "clocks": "true",
        "pgnInJson": "true"
    }
    headers = {
        "Accept": "application/x-ndjson",
        "User-Agent": "chess-historic-analytics/1.0"
    }

    games = []
    response = requests.get(url, params=params, headers=headers, stream=True)

    if response.status_code == 200:
        for line in response.iter_lines():
            if line:
                game = json.loads(line.decode("utf-8"))
                games.append(game)
    else:
        print(f"  ERROR: Status {response.status_code} for {username}")

    return games


# Fetch games for all players across tiers
all_games = []
player_tier_map = {}  # track which tier each game belongs to
successful_players = []

for username, tier in ALL_PLAYERS:
    print(f"[{tier}] Fetching games for {username}...")
    player_games = fetch_player_games(username, MAX_GAMES_PER_PLAYER)

    if player_games:
        print(f"  ✓ Retrieved {len(player_games)} games")
        for game in player_games:
            game["_tier"] = tier  # tag tier for later analysis
        all_games.extend(player_games)
        successful_players.append((username, tier, len(player_games)))
    else:
        print(f"  ✗ Skipped (not found or no rated games)")

print(f"\n{'='*50}")
print(f"Total games downloaded: {len(all_games)}")
print(f"Successful players: {len(successful_players)}/{len(ALL_PLAYERS)}")
print(f"\nBreakdown by tier:")
from collections import Counter
tier_counts = Counter()
for _, tier, count in successful_players:
    tier_counts[tier] += count
for tier, count in tier_counts.items():
    print(f"  {tier}: {count} games")

In [0]:
# ============================================================
# INGESTION VALIDATION
# ============================================================

print(f"Total games collected: {len(all_games)}")

# Count games per player
from collections import Counter
player_counts = Counter()
for game in all_games:
    white = game.get("players", {}).get("white", {}).get("user", {}).get("name", "unknown")
    black = game.get("players", {}).get("black", {}).get("user", {}).get("name", "unknown")
    player_counts[white] += 1
    player_counts[black] += 1

print("\nGames involving each player:")
for player, count in player_counts.most_common(10):
    print(f"  {player}: {count}")

# Show sample game structure
print("\nSample game keys:")
if all_games:
    print(f"  {list(all_games[0].keys())}")
    print(f"\nFirst game preview:")
    sample = all_games[0]
    print(f"  ID       : {sample.get('id')}")
    print(f"  Speed    : {sample.get('speed')}")
    print(f"  Status   : {sample.get('status')}")
    print(f"  Opening  : {sample.get('opening', {}).get('name')}")
    print(f"  ECO      : {sample.get('opening', {}).get('eco')}")
    print(f"  White    : {sample.get('players', {}).get('white', {}).get('user', {}).get('name')}")
    print(f"  Black    : {sample.get('players', {}).get('black', {}).get('user', {}).get('name')}")

In [0]:
# ============================================================
# BRONZE DATAFRAME
#
# Purpose:
# Convert raw Python game objects into a typed Spark DataFrame.
# Each game is stored as a raw JSON string with minimal metadata.
# No transformations or cleaning are performed.
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)
from pyspark.sql.functions import lit, current_timestamp

# Prepare bronze records: raw JSON + metadata
ingestion_time = datetime.now(timezone.utc)

bronze_records = []
for game in all_games:
    tier = game.pop("_tier", "unknown")  # remove internal tag
    bronze_records.append({
        "raw_event": json.dumps(game),
        "event_type": "historic_game",
        "game_id": game.get("id", "unknown"),
        "player_tier": tier,
        "source": "lichess_api",
        "ingested_at": ingestion_time
    })

# Define bronze schema
bronze_schema = StructType([
    StructField("raw_event", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("game_id", StringType(), False),
    StructField("player_tier", StringType(), False),
    StructField("source", StringType(), False),
    StructField("ingested_at", TimestampType(), False)
])

bronze_df = spark.createDataFrame(bronze_records, schema=bronze_schema)

print(f"Total bronze records: {bronze_df.count()}")

In [0]:
# ============================================================
# BRONZE DATA VALIDATION
# ============================================================

from pyspark.sql.functions import col, sum as _sum

print("Schema")
bronze_df.printSchema()

print(f"\nTotal Records : {bronze_df.count()}")

print("\nSource Distribution")
bronze_df.groupBy("source").count().show()

print("\nNull Check")
bronze_df.select(
    _sum(col("raw_event").isNull().cast("int")).alias("null_raw_event"),
    _sum(col("game_id").isNull().cast("int")).alias("null_game_id")
).show()

print("\nSample Records")
display(bronze_df.limit(10))

In [0]:
# ============================================================
# WRITE BRONZE DELTA TABLE
# ============================================================

if DEVELOPMENT_MODE:
    spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE}")

(
    bronze_df
    .write
    .format("delta")
    .mode("overwrite" if DEVELOPMENT_MODE else "append")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Successfully wrote {bronze_df.count()} records to {BRONZE_TABLE}")